# Alloy quick start

Alloy is a physics engine for robot learning: robots, cloth, liquids, cables and more in one scene. This notebook installs it, drops a box on a table, drapes a cloth over a stand and shows one rendered frame. It runs on the free CPU runtime in a few minutes; most of that is a one-time compile, cached afterwards.

Docs: https://alloy.argonrobotics.ai/docs/

In [ ]:
%pip install -q alloy-engine

In [ ]:
import alloy

print(alloy.__version__)

## 1. Drop a box

Sizes are full edge lengths in metres, masses in kilograms, and z points up. `sim.run` returns the recorded motion as numpy arrays.

In [ ]:
scene = alloy.Scene(dt=0.01)
scene.add_ground()
scene.add_box("table", size=(0.6, 0.6, 0.4), pos=(0, 0, 0.2), static=True)   # top at z = 0.4
scene.add_box("block", size=(0.05, 0.05, 0.05), pos=(0, 0, 0.6), mass=0.2)
sim = alloy.build(scene, device="cpu")
traj = sim.run(1.0, record=["block"])                       # 1 s; frame 0 is the start state
print(traj["block.pos"].shape)                              # (frames, worlds, 3)
print(traj["block.pos"][-1, 0].round(3), sim.check()["ok"])  # the block rests on the table

## 2. Drape a cloth

A cotton sheet falls onto a stand. Alloy picks a solver for each object; `sim.routes()` says which. On a CPU this cell takes a minute or two.

In [ ]:
scene = alloy.Scene(dt=0.01)
scene.add_ground()
scene.add_box("stand", size=(0.2, 0.2, 0.3), pos=(0, 0, 0.15), static=True)
scene.add_cloth("sheet", size=(0.5, 0.5), pos=(0, 0, 0.5), resolution=16, mass=0.1)
sim = alloy.build(scene, device="cpu")
sim.run(0.8)
print({name: r["route"] for name, r in sim.routes().items()})
print(sim.check()["ok"])

## 3. Look at it

`sim.render` writes one frame of the scene as it is now.

In [ ]:
from IPython.display import Image

sim.render(width=480, height=360, path="drape.png")
Image("drape.png")

## Next

- Quickstart and guides: https://alloy.argonrobotics.ai/docs/quickstart/
- With a GPU runtime (Runtime > Change runtime type > T4 GPU) drop `device="cpu"`: `alloy.build(scene)` uses the GPU when it finds one, and many worlds step together (`alloy.Scene(worlds=64, dt=0.01)`).
- Robots, liquids and cables: `python -m alloy.docs` prints where the docs and examples sit in the package.